In [1]:
#import libraries and set paths
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Dataset location (your local GLENDA folder) ---
DATA_RAW = Path(r"C:\Users\Admin\Endometriosis\Glenda")

# --- Project repo (where manifests will be saved) ---
PROJECT_ROOT = Path(r"C:\Users\Admin\Documents\glenda-lesion-localisation")
MANIFEST_DIR = PROJECT_ROOT / "data_manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset:   {DATA_RAW}")
print(f"Exists:    {DATA_RAW.exists()}")
print(f"Repo:      {PROJECT_ROOT}")
print(f"Manifests: {MANIFEST_DIR}")


Dataset:   C:\Users\Admin\Endometriosis\Glenda
Exists:    True
Repo:      C:\Users\Admin\Documents\glenda-lesion-localisation
Manifests: C:\Users\Admin\Documents\glenda-lesion-localisation\data_manifests


In [2]:
# explore the folder 
print("Top-level contents of the dataset folder:\n")
for p in sorted(DATA_RAW.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"  [{kind}] {p.name}")


Top-level contents of the dataset folder:

  [DIR ] Glenda_v1.5_classes


In [26]:
# explore the full tree
for root, dirs, files in os.walk(DATA_RAW):
    rel = Path(root).relative_to(DATA_RAW)
    depth = len(rel.parts)
    if depth > 3:
        dirs[:] = []
        continue
    indent = "  " * depth
    print(f"{indent}{rel}/  ({len(files)} files)")

./  (0 files)
  Glenda_v1.5_classes/  (6 files)
    Glenda_v1.5_classes\annots/  (373 files)
    Glenda_v1.5_classes\frames/  (373 files)
    Glenda_v1.5_classes\split_vis/  (1 files)
  no_pathology/  (0 files)
    no_pathology\frames/  (0 files)
      no_pathology\frames\v_2506_s_0-95/  (95 files)
      no_pathology\frames\v_2507_s_0-853/  (853 files)
      no_pathology\frames\v_2507_s_2320-3371/  (1051 files)
      no_pathology\frames\v_2607_s_2534-2971/  (437 files)
      no_pathology\frames\v_2690_s_154-252/  (98 files)
      no_pathology\frames\v_2736_s_19-698/  (679 files)
      no_pathology\frames\v_3312_s_0-278/  (278 files)
      no_pathology\frames\v_3312_s_1737-1823/  (86 files)
      no_pathology\frames\v_3315_s_1-237/  (236 files)
      no_pathology\frames\v_3528_s_1030-2231/  (1201 files)
      no_pathology\frames\v_3528_s_179-537/  (358 files)
      no_pathology\frames\v_3570_s_1247-1816/  (569 files)
      no_pathology\frames\v_3598_s_197-518/  (321 files)
      no_path

In [4]:
# count the image files 
extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff"]
image_files = []
for ext in extensions:
    image_files.extend(DATA_RAW.rglob(ext))

# De-duplicate in case a file matches two patterns
image_files = sorted(set(image_files))

print(f"Total image files found: {len(image_files)}")
print("\nFirst 25 file paths (relative to dataset root):")
for f in image_files[:25]:
    print(f"  {f.relative_to(DATA_RAW)}")

Total image files found: 746

First 25 file paths (relative to dataset root):
  Glenda_v1.5_classes\annots\c_100_v_(video_3062.mp4)_f_1458.png
  Glenda_v1.5_classes\annots\c_102_v_(video_3119.mp4)_f_1311.png
  Glenda_v1.5_classes\annots\c_102_v_(video_3119.mp4)_f_185.png
  Glenda_v1.5_classes\annots\c_102_v_(video_3124.mp4)_f_97.png
  Glenda_v1.5_classes\annots\c_102_v_(video_3130.mp4)_f_0.png
  Glenda_v1.5_classes\annots\c_105_v_(video_3214.mp4)_f_2954.png
  Glenda_v1.5_classes\annots\c_108_v_(video_3312.mp4)_f_619.png
  Glenda_v1.5_classes\annots\c_10_v_(video_182.mp4)_f_494.png
  Glenda_v1.5_classes\annots\c_10_v_(video_184.mp4)_f_22.png
  Glenda_v1.5_classes\annots\c_10_v_(video_185.mp4)_f_112.png
  Glenda_v1.5_classes\annots\c_10_v_(video_185.mp4)_f_176.png
  Glenda_v1.5_classes\annots\c_10_v_(video_186.mp4)_f_200.png
  Glenda_v1.5_classes\annots\c_10_v_(video_188.mp4)_f_455.png
  Glenda_v1.5_classes\annots\c_113_v_(video_3427.mp4)_f_211.png
  Glenda_v1.5_classes\annots\c_114_v_(v

In [5]:
# check for other existing zips 
zips = list(DATA_RAW.rglob("*.zip"))
print(f"Zip files found: {len(zips)}")
for z in zips:
    size_mb = z.stat().st_size / (1024 * 1024)
    print(f"  {z.relative_to(DATA_RAW)}  ({size_mb:.1f} MB)")


Zip files found: 0


In [10]:
# case id and video id extractors

# Accept both .jpg (image) and .png (mask)
FILENAME_PATTERN = re.compile(
    r"^c_(\d+)_v_\(video_(\d+)\.mp4\)_f_(\d+)\.(jpg|png)$",
    re.IGNORECASE,
)


def parse_filename(filepath: Path):
    """
    Return (case_id, video_id, frame_number, kind) or None if unparseable.
    kind is 'image' for .jpg and 'mask' for .png.
    """
    m = FILENAME_PATTERN.match(filepath.name)
    if not m:
        return None
    case_num, video_num, frame_num, ext = m.groups()
    ext = ext.lower()
    return (
        f"c_{case_num}",
        f"video_{video_num}",
        int(frame_num),
        "image" if ext == "jpg" else "mask",
    )


# Test
print(f"{'filename':55s}  {'case':7s}  {'video':12s}  {'frame':>6s}  kind")
print("-" * 95)
for f in image_files[:20]:
    parsed = parse_filename(f)
    if parsed is None:
        print(f"{f.name:55s}  PARSE FAILED")
    else:
        case_id, video_id, frame_num, kind = parsed
        print(f"{f.name:55s}  {case_id:7s}  {video_id:12s}  {frame_num:>6d}  {kind}")



filename                                                 case     video          frame  kind
-----------------------------------------------------------------------------------------------
c_100_v_(video_3062.mp4)_f_1458.png                      c_100    video_3062      1458  mask
c_102_v_(video_3119.mp4)_f_1311.png                      c_102    video_3119      1311  mask
c_102_v_(video_3119.mp4)_f_185.png                       c_102    video_3119       185  mask
c_102_v_(video_3124.mp4)_f_97.png                        c_102    video_3124        97  mask
c_102_v_(video_3130.mp4)_f_0.png                         c_102    video_3130         0  mask
c_105_v_(video_3214.mp4)_f_2954.png                      c_105    video_3214      2954  mask
c_108_v_(video_3312.mp4)_f_619.png                       c_108    video_3312       619  mask
c_10_v_(video_182.mp4)_f_494.png                         c_10     video_182        494  mask
c_10_v_(video_184.mp4)_f_22.png                          c_10     v

In [11]:
# verify all files are parseable
failures = [f for f in image_files if parse_filename(f) is None]
print(f"Total files:         {len(image_files)}")
print(f"Parse failures:      {len(failures)}")
print(f"Parse success rate:  {100 * (1 - len(failures) / len(image_files)):.2f}%")

if failures:
    print("\nFirst 10 failures:")
    for f in failures[:10]:
        print(f"  {f.name}")


Total files:         746
Parse failures:      0
Parse success rate:  100.00%


In [12]:
# count the images vs masks
from collections import Counter

kinds = Counter(parse_filename(f)[3] for f in image_files if parse_filename(f))
print("File kinds:")
for k, v in kinds.items():
    print(f"  {k:6s}: {v}")

# Pair up image + mask by base name (without extension)
images = {f.stem: f for f in image_files if f.suffix.lower() == ".jpg"}
masks  = {f.stem: f for f in image_files if f.suffix.lower() == ".png"}

print(f"\nUnique image stems: {len(images)}")
print(f"Unique mask stems:  {len(masks)}")

paired   = set(images) & set(masks)
img_only = set(images) - set(masks)
mask_only = set(masks) - set(images)

print(f"\nPaired (image + mask): {len(paired)}")
print(f"Image only (no mask):  {len(img_only)}")
print(f"Mask only (no image):  {len(mask_only)}")

if img_only:
    print(f"\nExamples of image-only: {sorted(img_only)[:5]}")
if mask_only:
    print(f"\nExamples of mask-only:  {sorted(mask_only)[:5]}")

File kinds:
  mask  : 373
  image : 373

Unique image stems: 373
Unique mask stems:  373

Paired (image + mask): 373
Image only (no mask):  0
Mask only (no image):  0


In [13]:
# look for negative frames
print("=== Top-level dataset contents ===\n")
for p in sorted(DATA_RAW.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"  [{kind}] {p.name}")

=== Top-level dataset contents ===

  [DIR ] Glenda_v1.5_classes


In [14]:
# explore the dataset
CLASSES_DIR = DATA_RAW / "Glenda_v1.5_classes"

print("Contents of Glenda_v1.5_classes:\n")
for p in sorted(CLASSES_DIR.iterdir()):
    if p.is_dir():
        n_files = sum(1 for _ in p.rglob("*.*"))
        print(f"  [DIR ] {p.name:30s}  ({n_files} files recursively)")
    else:
        print(f"  [FILE] {p.name}")

Contents of Glenda_v1.5_classes:

  [DIR ] annots                          (373 files recursively)
  [FILE] coco.json
  [FILE] colors.txt
  [FILE] config.json
  [DIR ] frames                          (373 files recursively)
  [FILE] label_colors.html
  [FILE] labels.txt
  [DIR ] split_vis                       (1 files recursively)
  [FILE] statistics_overall.csv


In [15]:
# read the label files 
CLASSES_DIR = DATA_RAW / "Glenda_v1.5_classes"

for name in ["labels.txt", "colors.txt"]:
    path = CLASSES_DIR / name
    print(f"=== {name} ===")
    if path.exists():
        print(path.read_text())
    else:
        print("  (missing)")
    print()

=== labels.txt ===
background
6.1.1.1_Endo-Peritoneum
6.1.1.2_Endo-Ovar
6.1.1.3_Endo-TIE
6.1.1.4_Endo-Uterus


=== colors.txt ===
0 0 0
190 62 204
93 89 254
145 210 138
238 236 50




In [16]:
# inspect coco.json
import json

coco_path = CLASSES_DIR / "coco.json"
with open(coco_path, "r") as f:
    coco = json.load(f)

print("Top-level keys in coco.json:")
for k, v in coco.items():
    if isinstance(v, list):
        print(f"  {k:15s}: list of {len(v)}")
    elif isinstance(v, dict):
        print(f"  {k:15s}: dict with {len(v)} entries")
    else:
        print(f"  {k:15s}: {type(v).__name__}")

Top-level keys in coco.json:
  info           : dict with 6 entries
  categories     : list of 4
  images         : list of 373
  annotations    : list of 628


In [17]:
# categories 
print("Categories:\n")
for cat in coco["categories"]:
    print(f"  id={cat['id']:3d}  name={cat.get('name', '?'):20s}  "
          f"supercategory={cat.get('supercategory', '-')}")

Categories:

  id=  1  name=6.1.1.1_Endo-Peritoneum  supercategory=6.1.1.1_Endo-Peritoneum
  id=  2  name=6.1.1.2_Endo-Ovar     supercategory=6.1.1.2_Endo-Ovar
  id=  3  name=6.1.1.3_Endo-TIE      supercategory=6.1.1.3_Endo-TIE
  id=  4  name=6.1.1.4_Endo-Uterus   supercategory=6.1.1.4_Endo-Uterus


In [18]:
# sample image entry
sample_img = coco["images"][0]
print("Sample image entry from coco.json:\n")
for k, v in sample_img.items():
    print(f"  {k}: {v}")

Sample image entry from coco.json:

  id: 0
  width: 640
  height: 360
  file_name: c_100_v_(video_3062.mp4)_f_1458.jpg
  path: frames/c_100_v_(video_3062.mp4)_f_1458.jpg
  license: None
  fickr_url: None
  coco_url: frames/c_100_v_(video_3062.mp4)_f_1458.jpg
  date_captured: None
  metadata: {'coco_url': 'frames/c_100_v_(video_3062.mp4)_f_1458.jpg'}


In [19]:
# sample annotation entry 
sample_ann = coco["annotations"][0]
print("Sample annotation entry from coco.json:\n")
for k, v in sample_ann.items():
    if k == "segmentation":
        # Truncate long polygon lists for readability
        if isinstance(v, list) and v and isinstance(v[0], list):
            print(f"  {k}: polygon with {len(v)} parts, "
                  f"first part has {len(v[0])} coords")
        else:
            print(f"  {k}: {type(v).__name__} (length {len(v) if hasattr(v, '__len__') else 'n/a'})")
    elif k == "bbox":
        print(f"  {k}: {v}  (x, y, w, h)")
    else:
        print(f"  {k}: {v}")

Sample annotation entry from coco.json:

  id: 0
  image_id: 0
  category_id: 4
  width: 640
  height: 360
  area: 904
  segmentation: polygon with 1 parts, first part has 84 coords
  bbox: [466, 264, 34, 31]  (x, y, w, h)
  metadata: {}
  color: #eeec32
  iscrowd: 0
  isbbox: False


In [20]:
# check image paths match reality

missing = []
for img in coco["images"]:
    fn = img["file_name"]
    # Try a few common layouts
    candidates = [
        CLASSES_DIR / fn,
        CLASSES_DIR / "frames" / fn,
        CLASSES_DIR / "frames" / Path(fn).name,
    ]
    if not any(c.exists() for c in candidates):
        missing.append(fn)

print(f"Total images in coco.json: {len(coco['images'])}")
print(f"Images not found on disk:  {len(missing)}")
if missing:
    print(f"\nFirst 5 missing:")
    for m in missing[:5]:
        print(f"  {m}")

Total images in coco.json: 373
Images not found on disk:  0


In [21]:
# build the frame manifest

# Reuse the filename parser from before
FILENAME_PATTERN = re.compile(
    r"^c_(\d+)_v_\(video_(\d+)\.mp4\)_f_(\d+)\.(jpg|png)$",
    re.IGNORECASE,
)

def parse_filename(name: str):
    m = FILENAME_PATTERN.match(name)
    if not m:
        return None
    c, v, f, _ = m.groups()
    return f"c_{c}", f"video_{v}", int(f)


cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
img_id_to_meta = {img["id"]: img for img in coco["images"]}

# Group annotations by image
anns_per_image = {}
for ann in coco["annotations"]:
    anns_per_image.setdefault(ann["image_id"], []).append(ann)

rows = []
for img_id, meta in tqdm(img_id_to_meta.items()):
    name = Path(meta["file_name"]).name
    parsed = parse_filename(name)
    if parsed is None:
        case_id, video_id, frame_num = "unknown", "unknown", -1
    else:
        case_id, video_id, frame_num = parsed

    anns = anns_per_image.get(img_id, [])
    cat_names = sorted({cat_id_to_name[a["category_id"]] for a in anns})
    n_annots = len(anns)
    total_area = sum(a.get("area", 0) for a in anns)

    rows.append({
        "image_id":   img_id,
        "file_name":  name,
        "case_id":    case_id,
        "video_id":   video_id,
        "frame_num":  frame_num,
        "width":      meta.get("width"),
        "height":     meta.get("height"),
        "n_annotations": n_annots,
        "classes":    ";".join(cat_names),
        "total_area": total_area,
    })

df = pd.DataFrame(rows)
print(f"Rows in manifest: {len(df)}")
print(f"Total annotations across all frames: {df['n_annotations'].sum()}")
print(f"\nUnique cases: {df['case_id'].nunique()}")
print(f"Unique videos: {df['video_id'].nunique()}")

  0%|          | 0/373 [00:00<?, ?it/s]

Rows in manifest: 373
Total annotations across all frames: 628

Unique cases: 102
Unique videos: 205


In [22]:
# class distribution summary
from collections import Counter

print("Annotations per class (instance level):\n")
cnt = Counter()
for ann in coco["annotations"]:
    cnt[cat_id_to_name[ann["category_id"]]] += 1
for name, n in cnt.most_common():
    print(f"  {name:15s}: {n}")

print("\nFrames containing each class:\n")
frame_cnt = Counter()
for ann in coco["annotations"]:
    frame_cnt[cat_id_to_name[ann["category_id"]]] += 1  # one per annotation; recalc below for frames

# Frames per class (unique image_id)
frames_per_class = {}
for ann in coco["annotations"]:
    cls = cat_id_to_name[ann["category_id"]]
    frames_per_class.setdefault(cls, set()).add(ann["image_id"])

for name, imgs in sorted(frames_per_class.items(), key=lambda x: -len(x[1])):
    print(f"  {name:15s}: {len(imgs)} frames")

Annotations per class (instance level):

  6.1.1.1_Endo-Peritoneum: 489
  6.1.1.3_Endo-TIE: 59
  6.1.1.2_Endo-Ovar: 54
  6.1.1.4_Endo-Uterus: 26

Frames containing each class:

  6.1.1.1_Endo-Peritoneum: 257 frames
  6.1.1.3_Endo-TIE: 55 frames
  6.1.1.2_Endo-Ovar: 53 frames
  6.1.1.4_Endo-Uterus: 17 frames


In [23]:
# save the manifest
frame_path = MANIFEST_DIR / "frame_metadata.csv"
df.to_csv(frame_path, index=False)
print(f"Saved: {frame_path}")

case_summary = (
    df.groupby(["case_id"])
      .agg(n_frames=("image_id", "nunique"),
           n_videos=("video_id", "nunique"),
           n_annotations=("n_annotations", "sum"))
      .reset_index()
      .sort_values("case_id")
)
case_path = MANIFEST_DIR / "case_manifest.csv"
case_summary.to_csv(case_path, index=False)
print(f"Saved: {case_path}")

print(f"\nCases: {len(case_summary)}")

Saved: C:\Users\Admin\Documents\glenda-lesion-localisation\data_manifests\frame_metadata.csv
Saved: C:\Users\Admin\Documents\glenda-lesion-localisation\data_manifests\case_manifest.csv

Cases: 102


In [24]:
# the missing negative frames 
# Look anywhere on disk for a no_pathology folder
search_roots = [
    DATA_RAW,
    DATA_RAW.parent,
    Path(r"C:\Users\Admin\Endometriosis"),
]

found = False
for root in search_roots:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if p.is_dir() and "no_patholog" in p.name.lower():
            print(f"Found: {p}")
            found = True

if not found:
    print("No no_pathology folder found on disk.")
    print("The negative frames must be downloaded separately.")
    print("Go to: https://ftp.itec.aau.at/datasets/GLENDA/index.html")
    print("Look for a download labelled 'no pathology' or 'negative'.")

No no_pathology folder found on disk.
The negative frames must be downloaded separately.
Go to: https://ftp.itec.aau.at/datasets/GLENDA/index.html
Look for a download labelled 'no pathology' or 'negative'.


# SUMMARY
# GLENDA v1.5 Audit Summary

## Dataset location
`C:\Users\Admin\Endometriosis\Glenda\Glenda_v1.5_classes`

## Format
COCO-exported release. Single authoritative `coco.json` with 373 images
and 628 annotations. Masks in `annots/`, RGB frames in `frames/`.

## Counts
- Total images:                 373
- Total annotations:            628
- Unique cases:                 102
- Unique videos:                205
- Image/mask pairing failures:  0

## Class distribution
| Class | Dataset name                  | Annotations | Frames | Cases |
|-------|-------------------------------|-------------|--------|-------|
| Peritoneum | 6.1.1.1_Endo-Peritoneum   | 489         | 257    | TBD   |
| Ovary      | 6.1.1.2_Endo-Ovar         | 54          | 53     | TBD   |
| TIE (DIE)  | 6.1.1.3_Endo-TIE          | 59          | 55     | TBD   |
| Uterus     | 6.1.1.4_Endo-Uterus       | 26          | 17     | TBD   |

## Terminology note
The dataset labels Deep Infiltrating Endometriosis as `TIE` (German:
Tief Infiltrierende Endometriose). This is the same class as "DIE" in
the published GLENDA literature.

## Negative frames
13,438 lesion-free frames from 27 cases are released as a **separate
download** and are not included in this folder. 

## Naming convention
- Filename pattern: `c_<case>_v_(video_<video>.mp4)_f_<frame>.<jpg|png>`
- `.jpg` = RGB image, `.png` = colour-coded mask
- Case ID extracted via regex from filename
- Class labels read from `coco.json`, not from folder names

## Open questions
- Are the 27 negative cases disjoint from the 102 annotated cases?
  (To verify once negatives are downloaded.)

In [25]:
# TBD for cases per class 
per_class_cases = {}
for ann in coco["annotations"]:
    cls = cat_id_to_name[ann["category_id"]]
    img = img_id_to_meta[ann["image_id"]]
    parsed = parse_filename(Path(img["file_name"]).name)
    case = parsed[0] if parsed else "unknown"
    per_class_cases.setdefault(cls, set()).add(case)

print("Cases per class:")
for cls, cases in sorted(per_class_cases.items(), key=lambda x: -len(x[1])):
    print(f"  {cls:30s}: {len(cases)}")

Cases per class:
  6.1.1.1_Endo-Peritoneum       : 84
  6.1.1.3_Endo-TIE              : 29
  6.1.1.2_Endo-Ovar             : 27
  6.1.1.4_Endo-Uterus           : 11
